# Task 6: Running the Model (Run summary on CPU)

Summary for the Task 4 / Task 6 Legal-BERT runs (unweighted reference vs. pos_weight, seeds 42 / 123 / 2026).

```
notebooks/october_task4_outputs/
├── full_unweighted
│   ├── results_unweighted.json
│   ├── unweighted_Legal-BERT_<seed>.png                    # for each seed
│   ├── unweighted_Legal-BERT_<seed>_val_outputs.npz        # for each seed
│   └── unweighted_Legal-BERT_<seed>_val_scores.npz         # for each seed
├── full_weighted
│   ├── results_weighted.json
    │   ├── weighted_Legal-BERT_<seed>.png                  # for each seed
│   ├── weighted_Legal-BERT_<seed>_val_outputs.npz          # for each seed
│   └── weighted_Legal-BERT_<seed>_val_scores.npz           # for each seed
├── train_chunk_ids.json
└── val_chunk_ids.json
```

## What's Not on GitHub
- tokenizer - *located in: MyDrive/october_task4_outputs (note this is a separate folder from '[BTT] Accenture 1O)'*
- thresholds - *uses 0.5 by default, selection is in Task 8*

In [1]:
from pathlib import Path
import json

config_path = Path("october_task1_outputs/task1_frozen_model_specification.json")
with open(config_path, 'r', encoding='utf-8') as file:
    frozen_config = json.load(file)

MODEL_ID = frozen_config["encoder"]["primary_model_id"]
MODEL_NAME = frozen_config["encoder"]["primary_name"]
SEEDS = frozen_config["training"]["random_seeds"]

FINAL_IDS = frozen_config["categories"]

OUTPUT_DIR = Path("october_task6_outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
TASK4_OUTPUT_DIR = Path("october_task4_outputs")
UNWEIGHTED = TASK4_OUTPUT_DIR / "full_unweighted"
WEIGHTED = TASK4_OUTPUT_DIR / "full_weighted"

print(f"Model: {frozen_config["encoder"]["primary_name"]}")
print(f"Seeds: {SEEDS}")

Model: Legal-BERT
Seeds: [42, 123, 2026]


## Loading Run Data

From the Task 4 notebook, a total of 6 runs occurred: one for each seed, both unweighted and weighted (using positive_weights to address class imbalance)

In [2]:
import numpy as np

def load_run(run, seed):
    d = TASK4_OUTPUT_DIR / f"full_{run}"
    out = np.load(d / f"{run}_Legal-BERT_seed{seed}_val_outputs.npz")
    sc  = np.load(d / f"{run}_Legal-BERT_seed{seed}_val_scores.npz")
    return {k: out[k] for k in out.files}, {k: sc[k] for k in sc.files}

runs = {(run, s): load_run(run, s)
        for run in ("unweighted", "weighted") for s in SEEDS}

# Row alignment: rows follow val_chunk_ids.json order
val_ids = json.load(open(TASK4_OUTPUT_DIR / "val_chunk_ids.json"))

# Per-run summary metrics
res_u = json.load(open(TASK4_OUTPUT_DIR / "full_unweighted" / "results_unweighted.json"))
res_w = json.load(open(TASK4_OUTPUT_DIR / "full_weighted" / "results_weighted.json"))


print("unweighted: ", {s: r["status"] for s, r in list(res_u.values())[0].items()})
print("weighted: ", {s: r["status"] for s, r in list(res_w.values())[0].items()})

unweighted:  {'42': 'completed', '123': 'completed', '2026': 'completed'}
weighted:  {'42': 'completed', '123': 'completed', '2026': 'completed'}


## Contents

The `full_unweighted` folder contains the following for each seed:
- val_outputs: labels, logits
- val_scores: chunk_id, logits

The `full_weighted` folder contains the following for each seed:
- val_outputs: chunk_id, labels, logits
- val_scores: chunk_id, logits

In [3]:
seed = SEEDS[0]

def describe(name, d):
    print(f"\n{name}")
    for k, a in d.items():
        print(f"  {k:<10} shape={a.shape!s:<12} dtype={a.dtype}")

u_out, u_sc = runs[("unweighted", seed)]
w_out, w_sc = runs[("weighted", seed)]

print("unweighted outputs keys:", sorted(u_out))
print("unweighted scores keys: ", sorted(u_sc))
print("")
print("weighted outputs keys:  ", sorted(w_out))
print("weighted scores keys:   ", sorted(w_sc))

unweighted outputs keys: ['val_labels', 'val_logits']
unweighted scores keys:  ['chunk_ids', 'val_logits']

weighted outputs keys:   ['chunk_ids', 'val_labels', 'val_logits']
weighted scores keys:    ['chunk_ids', 'val_logits']


More detail:

In [4]:
describe(f"unweighted | seed {seed} | val_outputs", u_out)
describe(f"unweighted | seed {seed} | val_scores",  u_sc)
describe(f"weighted   | seed {seed} | val_outputs", w_out)
describe(f"weighted   | seed {seed} | val_scores",  w_sc)

print("\nlen(val_ids):", len(val_ids))


unweighted | seed 42 | val_outputs
  val_logits shape=(2873, 10)   dtype=float32
  val_labels shape=(2, 2873, 10) dtype=int64

unweighted | seed 42 | val_scores
  val_logits shape=(2873, 10)   dtype=float32
  chunk_ids  shape=(2873,)      dtype=<U25

weighted   | seed 42 | val_outputs
  val_logits shape=(2873, 10)   dtype=float32
  val_labels shape=(2, 2873, 10) dtype=int64
  chunk_ids  shape=(2873,)      dtype=<U25

weighted   | seed 42 | val_scores
  val_logits shape=(2873, 10)   dtype=float32
  chunk_ids  shape=(2873,)      dtype=<U25

len(val_ids): 2873


## Alignment Check

In [6]:
def check(name, cond):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}")
    return cond

seed = SEEDS[0]
val = np.array(val_ids)
N = len(val)
print("N = ", N)

# 1. Row counts match val_chunk_ids.json (val_labels has rows on axis 1)
check("unweighted outputs logits rows == N", u_out["val_logits"].shape[0] == N)
check("weighted outputs logits rows   == N", w_out["val_logits"].shape[0] == N)
check("unweighted scores logits rows  == N", u_sc["val_logits"].shape[0] == N)
check("weighted scores logits rows    == N", w_sc["val_logits"].shape[0] == N)
check("unweighted val_labels rows     == N", u_out["val_labels"].shape[1] == N)
check("weighted val_labels rows       == N", w_out["val_labels"].shape[1] == N)

# 2. Chunk order matches val_chunk_ids.json
check("unweighted scores chunk_ids == val_ids", np.array_equal(u_sc["chunk_ids"], val))
check("weighted scores chunk_ids   == val_ids", np.array_equal(w_sc["chunk_ids"], val))
check("weighted outputs chunk_ids  == val_ids", np.array_equal(w_out["chunk_ids"], val))

# 3. Both runs share the same labels/mask and chunk order
check("val_labels identical across runs", np.array_equal(u_out["val_labels"], w_out["val_labels"]))
check("chunk_ids identical across runs",  np.array_equal(u_sc["chunk_ids"], w_sc["chunk_ids"]))

# 4. Scores logits == outputs logits within each run
check("unweighted scores logits ~= outputs logits", np.allclose(u_sc["val_logits"], u_out["val_logits"], atol=1e-4))
check("weighted scores logits ~= outputs logits",   np.allclose(w_sc["val_logits"], w_out["val_logits"], atol=1e-4))

# 5. Scores are label-free
check("unweighted scores have no labels", "val_labels" not in u_sc)
check("weighted scores have no labels",   "val_labels" not in w_sc)

# 6. Head width = 10 categories
check("10 output columns", u_out["val_logits"].shape[1] == 10 and w_out["val_logits"].shape[1] == 10)
check("len(FINAL_IDS) == 10", len(FINAL_IDS) == 10)

# 7. No train/val leakage
train_ids = set(json.load(open(TASK4_OUTPUT_DIR / "train_chunk_ids.json")))
check("train and val chunk ids disjoint", train_ids.isdisjoint(set(val_ids)))

N =  2873
[PASS] unweighted outputs logits rows == N
[PASS] weighted outputs logits rows   == N
[PASS] unweighted scores logits rows  == N
[PASS] weighted scores logits rows    == N
[PASS] unweighted val_labels rows     == N
[PASS] weighted val_labels rows       == N
[PASS] unweighted scores chunk_ids == val_ids
[PASS] weighted scores chunk_ids   == val_ids
[PASS] weighted outputs chunk_ids  == val_ids
[PASS] val_labels identical across runs
[PASS] chunk_ids identical across runs
[PASS] unweighted scores logits ~= outputs logits
[PASS] weighted scores logits ~= outputs logits
[PASS] unweighted scores have no labels
[PASS] weighted scores have no labels
[PASS] 10 output columns
[PASS] len(FINAL_IDS) == 10
[PASS] train and val chunk ids disjoint


True

# Label-Free Scorer